In [ ]:

"""
Free Recall Experiment  -  Human Memory Mini Project (DTU 02458/02464)
=====================================================================

Replicates the classic free-recall effects from Sections 1.5-1.6 of the
lecture notes across four interleaved conditions:

    1. standard   2 s / item, immediate recall            -> primacy + recency
    2. fast        0.5 s / item, immediate recall          -> primacy shrinks, recency ~flat
    3. filled      2 s / item, then a WM distractor, recall-> recency killed, primacy stays
    4. pause       2 s / item, then an EMPTY delay, recall -> recency drops much less than 'filled'

The 'filled' and 'pause' delays are the SAME duration so the two conditions
differ only in whether working memory is occupied (interference) or merely
time passes (decay).

Data is APPENDED across runs (never overwritten). Run number is auto-detected
from the existing data file, so running the script a second time for the same
participant just adds run 2, run 3, ... to the same files.

Standard library only (tkinter + csv) - nothing to pip install. Hand this file
in with the report.

Platform: this build is tuned for Windows (Segoe UI font). It also runs on
macOS. The window is brought to the front and given keyboard focus at launch so
the entry boxes accept typing straight away.
"""

import csv
import os
import random
import time
import tkinter as tk
from tkinter import simpledialog, font as tkfont

# =====================================================================
# CONFIG  -  everything you might tweak lives here
# =====================================================================
# --- the ONE switch you flip ---
PILOT = True                        # True  = short pilot: 1 rep/condition, data -> pilot_data/ (kept
                                    #         separate so it NEVER mixes into your real data).
                                    # False = real experiment: 10 reps/condition, data -> data/.

LIST_LENGTH               = 13      # words presented per trial (above WM capacity)
REPS_PER_CONDITION_PER_RUN = 1 if PILOT else 10   # 10 -> two runs give 20 reps/condition
STANDARD_RATE_S           = 2.0     # seconds per item, standard rate
FAST_RATE_S               = 0.5     # seconds per item, fast rate
ISI_FRACTION              = 0.2     # fraction of each item's time shown as a blank gap
DELAY_S                   = 25      # duration of BOTH the filled and pause delay (MUST match).
                                    # This is a fixed window, NOT "time to finish the task" - the
                                    # distractor is open-ended and gets cut off here. ~15s shrinks
                                    # recency, ~30s (Glanzer & Cunitz 1966) abolishes it. 25 is a
                                    # solid middle ground; use 30 for the textbook-clean effect.
DATA_DIR                  = "pilot_data" if PILOT else "data"   # kept separate so pilot != real data

CONDITIONS = ["standard", "fast", "filled", "pause"]

# 261 concrete, imageable nouns (nouns only, mostly 1-2 syllables). A larger pool
# keeps word reuse low across the ~80 trials (about 4x per word instead of ~10x with
# 100), which cuts proactive interference and cross-trial intrusions. Random lists are
# sampled from this pool, mirroring how studies draw from the Toronto Word Pool (1080).
WORD_POOL = [
    "book", "chair", "cloud", "stone", "river", "bread", "clock", "glass", "chain", "horse",
    "plant", "spoon", "brick", "flame", "sword", "coin", "wheel", "ghost", "throne", "nail",
    "leaf", "drum", "rope", "cage", "mask", "torch", "shell", "blade", "frame", "crown",
    "knife", "ladder", "mirror", "bottle", "pencil", "candle", "basket", "pillow", "window", "hammer",
    "jacket", "garden", "rocket", "tunnel", "engine", "castle", "forest", "bridge", "anchor", "pocket",
    "saddle", "kettle", "marble", "ribbon", "needle", "feather", "lantern", "thunder", "balloon", "blanket",
    "compass", "harvest", "machine", "whistle", "biscuit", "curtain", "monster", "planet", "island", "farmer",
    "button", "camera", "finger", "guitar", "helmet", "insect", "jungle", "magnet", "napkin", "orange",
    "parrot", "rabbit", "ticket", "violin", "walnut", "zebra", "apron", "cabin", "dagger", "eagle",
    "fabric", "goblet", "hedge", "ivory", "kernel", "lemon", "meadow", "nozzle", "oyster", "pebble",
    "tiger", "camel", "donkey", "beaver", "falcon", "salmon", "turtle", "spider", "lizard", "panther",
    "walrus", "badger", "pigeon", "sparrow", "buffalo", "leopard", "dolphin", "penguin", "squirrel", "raccoon",
    "moose", "ferret", "weasel", "lobster", "beetle", "cricket", "hornet", "snail", "otter", "hamster",
    "elbow", "ankle", "thumb", "wrist", "knuckle", "eyebrow", "spine", "heel", "jaw", "apple",
    "cherry", "carrot", "potato", "pepper", "pickle", "muffin", "waffle", "noodle", "pretzel", "pancake",
    "sausage", "pumpkin", "cabbage", "radish", "turnip", "almond", "apricot", "coconut", "mango", "melon",
    "olive", "peanut", "raisin", "celery", "boulder", "canyon", "glacier", "valley", "desert", "swamp",
    "prairie", "volcano", "breeze", "blossom", "acorn", "cactus", "fern", "moss", "petal", "branch",
    "gravel", "puddle", "ripple", "wrench", "shovel", "chisel", "pliers", "drill", "rake", "hook",
    "clamp", "trowel", "mallet", "drawer", "teapot", "saucer", "bucket", "broom", "sponge", "towel",
    "hanger", "thimble", "spatula", "ladle", "whisk", "colander", "corkscrew", "table", "bench", "cottage",
    "tower", "chimney", "fence", "gate", "balcony", "attic", "cellar", "hallway", "pillar", "glove",
    "scarf", "mitten", "sandal", "sweater", "trousers", "blouse", "bonnet", "cloak", "collar", "buckle",
    "zipper", "sleeve", "wagon", "tractor", "sleigh", "canoe", "kayak", "scooter", "trolley", "carriage",
    "glider", "chariot", "raft", "trumpet", "banjo", "flute", "harp", "cello", "tuba", "fiddle",
    "bagpipe", "tambourine", "harmonica", "telescope", "umbrella", "trophy", "crayon", "domino", "puzzle", "kite",
    "envelope",
]

# Data files (one combined file each, participant is a column -> easy to pool)
MAIN_FILE       = "freerecall_main.csv"        # one row per presented word
INTRUSION_FILE  = "freerecall_intrusions.csv"  # words recalled that were not in the list
WM_FILE         = "freerecall_wm.csv"          # counting-backwards compliance log

MAIN_HEADER      = ["participant", "run", "timestamp", "trial", "condition",
                    "rate_s", "delay_s", "serial_position", "item", "recalled"]
INTRUSION_HEADER = ["participant", "run", "timestamp", "trial", "condition", "intruded_word"]
WM_HEADER        = ["participant", "run", "timestamp", "trial", "start_number",
                    "typed_sequence", "n_correct_steps"]


# =====================================================================
# PURE LOGIC  (no GUI - unit-testable)
# =====================================================================
def build_trials(seed=None):
    """Return a shuffled list of trials for one run. Each trial is a dict with
    its condition and a freshly sampled list of LIST_LENGTH words."""
    rng = random.Random(seed)
    trials = []
    for cond in CONDITIONS:
        for _ in range(REPS_PER_CONDITION_PER_RUN):
            words = rng.sample(WORD_POOL, LIST_LENGTH)
            trials.append({"condition": cond, "words": words})
    rng.shuffle(trials)
    return trials


def rate_for(condition):
    return FAST_RATE_S if condition == "fast" else STANDARD_RATE_S


def delay_for(condition):
    """Post-list delay in seconds and its kind."""
    if condition == "filled":
        return DELAY_S, "filled"
    if condition == "pause":
        return DELAY_S, "pause"
    return 0, "none"


def score_trial(presented_words, recalled_words):
    """presented_words: list in serial order. recalled_words: set/list the
    participant produced. Returns (per_position_rows, intrusions)."""
    presented_set = set(presented_words)
    recalled_set = set(recalled_words)
    rows = []
    for pos, item in enumerate(presented_words, start=1):
        rows.append((pos, item, 1 if item in recalled_set else 0))
    intrusions = [w for w in recalled_words if w not in presented_set]
    return rows, intrusions


def count_correct_steps(start_number, typed_numbers):
    """Count how many consecutive entries went DOWN BY 3, regardless of where the
    participant anchored. Measures 'did they actually subtract 3 each step', so a
    correct run starting at start-3 (e.g. 135, 132, 129 from 138) scores full marks
    just like one starting at start. A gap that isn't 3 (or a repeat) breaks it."""
    n = 0
    for a, b in zip(typed_numbers, typed_numbers[1:]):
        if a - b == 3:
            n += 1
        else:
            break
    return n


def detect_run_number(data_dir, participant):
    """Next run number for this participant = (max existing run) + 1."""
    path = os.path.join(data_dir, MAIN_FILE)
    if not os.path.exists(path):
        return 1
    max_run = 0
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("participant") == participant:
                try:
                    max_run = max(max_run, int(row.get("run", 0)))
                except ValueError:
                    pass
    return max_run + 1


def ensure_headers(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    for fname, header in ((MAIN_FILE, MAIN_HEADER),
                          (INTRUSION_FILE, INTRUSION_HEADER),
                          (WM_FILE, WM_HEADER)):
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path):
            with open(path, "w", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow(header)


def append_rows(data_dir, fname, rows):
    with open(os.path.join(data_dir, fname), "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)


# =====================================================================
# GUI / EXPERIMENT FLOW
# =====================================================================
class Experiment:
    def __init__(self, root, participant, run):
        self.root = root
        self.participant = participant
        self.run = run
        self.ts = time.strftime("%Y-%m-%d %H:%M:%S")
        self.trials = build_trials()
        self.trial_index = 0

        self.root.title("Free Recall Experiment")
        self.root.configure(bg="black")
        self.root.attributes("-fullscreen", True)
        self.root.bind("<Escape>", lambda e: self.quit_early())

        self.big   = tkfont.Font(family="Segoe UI", size=54, weight="bold")
        self.mid   = tkfont.Font(family="Segoe UI", size=28)
        self.small = tkfont.Font(family="Segoe UI", size=18)

        self.canvas = tk.Frame(root, bg="black")
        self.canvas.pack(expand=True, fill="both")

        self.show_welcome()

    # ---- helpers ----
    def clear(self):
        for w in self.canvas.winfo_children():
            w.destroy()

    def centered_label(self, text, fnt, fg="white"):
        lbl = tk.Label(self.canvas, text=text, font=fnt, fg=fg, bg="black",
                       justify="center", wraplength=self.root.winfo_screenwidth() - 200)
        lbl.place(relx=0.5, rely=0.5, anchor="center")
        return lbl

    # ---- screens ----
    def show_welcome(self):
        self.clear()
        msg = (f"Free Recall Experiment\n\n"
               f"Participant: {self.participant}     Run: {self.run}\n"
               f"{len(self.trials)} lists this run\n\n"
               "You will see a sequence of words, one at a time.\n"
               "Try to remember as many as you can, in any order.\n"
               "Sometimes there will be a short task or pause before you recall.\n\n"
               "Press SPACE to begin.  (Esc quits and saves what you've done.)")
        self.centered_label(msg, self.mid)
        self.root.bind("<space>", self.start_next_trial)

    def start_next_trial(self, event=None):
        self.root.unbind("<space>")
        if self.trial_index >= len(self.trials):
            return self.show_goodbye()
        self.current = self.trials[self.trial_index]
        self.presented = self.current["words"]
        self.cond = self.current["condition"]
        self.show_fixation_then_present()

    def show_fixation_then_present(self):
        self.clear()
        self.centered_label("+", self.big)
        self.root.after(1000, self.present_words)

    def present_words(self, i=0):
        self.clear()
        rate_ms = int(rate_for(self.cond) * 1000)
        on_ms  = int(rate_ms * (1 - ISI_FRACTION))
        off_ms = rate_ms - on_ms
        if i < len(self.presented):
            self.centered_label(self.presented[i], self.big)
            # word on, then blank gap, then next word
            self.root.after(on_ms, lambda: (self.clear(),
                                             self.root.after(off_ms, lambda: self.present_words(i + 1))))
        else:
            self.after_list()

    def after_list(self):
        _, kind = delay_for(self.cond)
        if kind == "filled":
            self.run_wm_task()
        elif kind == "pause":
            self.run_pause()
        else:
            self.show_recall()

    # ---- delay: empty pause ----
    def run_pause(self):
        self.clear()
        self.centered_label("+", self.big)
        self.root.after(DELAY_S * 1000, self.show_recall)

    # ---- delay: filled WM distractor (count backwards by 3, type each) ----
    def run_wm_task(self):
        self.clear()
        self.wm_start = random.randint(100, 999)
        self.wm_typed = []
        tk.Label(self.canvas,
                 text=f"Start at {self.wm_start} and SUBTRACT 3 each time.\n"
                      f"So:  {self.wm_start - 3},  {self.wm_start - 6},  {self.wm_start - 9},  ...\n\n"
                      "Type each number and press Enter.\n"
                      "Keep going the WHOLE time, until the screen changes.",
                 font=self.mid, fg="white", bg="black", justify="center").place(relx=0.5, rely=0.32, anchor="center")
        self.wm_entry = tk.Entry(self.canvas, font=self.big, justify="center", width=8)
        self.wm_entry.place(relx=0.5, rely=0.55, anchor="center")
        self.wm_entry.focus_set()
        self.wm_echo = tk.Label(self.canvas, text="", font=self.small, fg="#8888ff", bg="black")
        self.wm_echo.place(relx=0.5, rely=0.68, anchor="center")
        self.wm_entry.bind("<Return>", self.wm_submit)
        self.root.after(DELAY_S * 1000, self.end_wm_task)

    def wm_submit(self, event=None):
        txt = self.wm_entry.get().strip()
        self.wm_entry.delete(0, tk.END)
        if txt.lstrip("-").isdigit():
            self.wm_typed.append(int(txt))
            self.wm_echo.config(text="  ".join(str(v) for v in self.wm_typed[-12:]))

    def end_wm_task(self):
        n_correct = count_correct_steps(self.wm_start, self.wm_typed)
        append_rows(DATA_DIR, WM_FILE, [[
            self.participant, self.run, self.ts, self.trial_index + 1,
            self.wm_start, " ".join(str(v) for v in self.wm_typed), n_correct,
        ]])
        self.show_recall()

    # ---- recall with pool-restricted autocomplete ----
    def show_recall(self):
        self.clear()
        self.recalled = []
        tk.Label(self.canvas, text="Recall the words - any order.\n"
                                    "Type, pick from the list, Enter to add. Click DONE when finished.",
                 font=self.mid, fg="white", bg="black", justify="center").place(relx=0.5, rely=0.10, anchor="center")

        self.entry = tk.Entry(self.canvas, font=self.mid, justify="center", width=20)
        self.entry.place(relx=0.5, rely=0.24, anchor="center")
        self.entry.focus_set()
        self.entry.bind("<KeyRelease>", self.update_suggestions)
        self.entry.bind("<Return>", self.add_from_entry)
        self.entry.bind("<Down>", lambda e: (self.listbox.focus_set(),
                                             self.listbox.selection_set(0)))

        self.listbox = tk.Listbox(self.canvas, font=self.mid, width=20, height=6)
        self.listbox.place(relx=0.5, rely=0.45, anchor="center")
        self.listbox.bind("<Return>", self.add_from_listbox)
        self.listbox.bind("<Double-Button-1>", self.add_from_listbox)

        self.chosen_lbl = tk.Label(self.canvas, text="Your words: (none yet)", font=self.small,
                                   fg="#88ff88", bg="black", justify="center",
                                   wraplength=self.root.winfo_screenwidth() - 300)
        self.chosen_lbl.place(relx=0.5, rely=0.70, anchor="center")

        tk.Button(self.canvas, text="DONE", font=self.mid, command=self.finish_recall,
                  bg="#333333", fg="white", padx=20, pady=8).place(relx=0.5, rely=0.86, anchor="center")

        self.update_suggestions()

    def update_suggestions(self, event=None):
        typed = self.entry.get().strip().lower()
        self.listbox.delete(0, tk.END)
        matches = [w for w in WORD_POOL
                   if w.startswith(typed) and w not in self.recalled] if typed else []
        for w in matches[:20]:
            self.listbox.insert(tk.END, w)

    def _add_word(self, word):
        if word in WORD_POOL and word not in self.recalled:
            self.recalled.append(word)
            self.chosen_lbl.config(text="Your words: " + ", ".join(self.recalled))
        self.entry.delete(0, tk.END)
        self.update_suggestions()
        self.entry.focus_set()

    def add_from_entry(self, event=None):
        typed = self.entry.get().strip().lower()
        if typed in WORD_POOL:
            self._add_word(typed)
        elif self.listbox.size() > 0:          # accept the top suggestion
            self._add_word(self.listbox.get(0))

    def add_from_listbox(self, event=None):
        sel = self.listbox.curselection()
        if sel:
            self._add_word(self.listbox.get(sel[0]))

    def finish_recall(self):
        rows, intrusions = score_trial(self.presented, self.recalled)
        main_rows = [[self.participant, self.run, self.ts, self.trial_index + 1, self.cond,
                      rate_for(self.cond), delay_for(self.cond)[0], pos, item, rec]
                     for (pos, item, rec) in rows]
        append_rows(DATA_DIR, MAIN_FILE, main_rows)
        if intrusions:
            append_rows(DATA_DIR, INTRUSION_FILE,
                        [[self.participant, self.run, self.ts, self.trial_index + 1, self.cond, w]
                         for w in intrusions])
        self.trial_index += 1
        self.show_break_or_next()

    def show_break_or_next(self):
        self.clear()
        done, total = self.trial_index, len(self.trials)
        self.centered_label(f"List {done} of {total} complete.\n\nPress SPACE for the next list.", self.mid)
        self.root.bind("<space>", self.start_next_trial)

    def show_goodbye(self):
        self.clear()
        self.centered_label("All done - thank you!\n\nData saved. You can close the window.", self.mid)

    def quit_early(self):
        self.root.destroy()


def main():
    root = tk.Tk()
    root.withdraw()
    participant = simpledialog.askstring("Participant", "Enter participant ID (e.g. student number):")
    if not participant:
        return
    participant = participant.strip()
    ensure_headers(DATA_DIR)
    run = detect_run_number(DATA_DIR, participant)
    root.deiconify()
    Experiment(root, participant, run)
    # Bring the fullscreen window to the front and grab keyboard focus so the
    # entry boxes accept typing without needing a click first.
    root.lift()
    root.after(50, root.focus_force)
    root.mainloop()


if __name__ == "__main__":
    main()

: 